# v4.1 Adaptive Reasoning System — Training Pipeline

Follows the staged training order:

| Phase | What | Description |
|-------|------|-------------|
| 1 | Solvers & Verifier | Make each mode work individually |
| 2 | Rule-based router | Get the full loop running end-to-end |
| 3 | JEPA-lite world model | Predict latent next state, validity, score gain, cost |
| 4 | Router EBM | Ranking loss on good vs bad reasoning actions |
| 5 | Top-k routing | Evaluate multiple candidates under verification |
| 6 | Stronger LLM usage | Code gen, critique, solver formulation, repair |

This notebook covers Phases 2–4 on a single A100/H100.

In [ ]:
# Mount Drive & install deps
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_ROOT = '/content/drive/MyDrive/agi'
sys.path.insert(0, PROJECT_ROOT)

# Pin ortools<9.12 to avoid protobuf>=6 conflict with Colab's tensorflow/google-ai
!pip install -q torch transformers accelerate bitsandbytes "ortools>=9.7,<9.12" faiss-cpu pydantic rich numpy

In [ ]:
import torch
import json
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Phase 2: Rule-Based Router — Collect Rollouts

Run the full loop with the rule-based router on synthetic problems
to populate the replay buffer with reasoning trajectories.

In [ ]:
from v4_1_reasoning_system.orchestration.controller import ReasoningController
from v4_1_reasoning_system.world_model.encoder import StateEncoder
from v4_1_reasoning_system.world_model.predictor import TransitionPredictor, ActionEncoder
from v4_1_reasoning_system.world_model.aux_heads import AuxiliaryHeads
from v4_1_reasoning_system.router.ebm_router import EBMRouter
from v4_1_reasoning_system.training.collect_rollouts import RolloutCollector

LATENT_DIM = 128
ACTION_DIM = 32

# Initialize controller with rule-based routing
controller = ReasoningController(
    use_learned_router=False,
    max_iterations=5,
    max_time_seconds=60.0,
    device=device,
    latent_dim=LATENT_DIM,
    action_dim=ACTION_DIM,
)
print('Controller ready (rule-based routing)')

In [ ]:
# Generate synthetic benchmark suite
problems = RolloutCollector.generate_benchmark_suite(
    n_planning=20,
    n_scheduling=20,
    n_optimization=20,
    n_coding=0,  # Skip coding for now (needs LLM)
)
print(f'Generated {len(problems)} benchmark problems')

In [ ]:
# Collect rollouts
collector = RolloutCollector(controller)
results = collector.collect(problems, verbose=True)

# Summary
successes = sum(1 for r in results if r['valid'])
print(f'\nResults: {successes}/{len(results)} valid solutions')
print(f'Avg score: {sum(r["score"] for r in results) / len(results):.3f}')
print(f'Avg iterations: {sum(r["iterations"] for r in results) / len(results):.1f}')

In [ ]:
# Check replay buffer
stats = controller.replay_buffer.statistics()
print(f'Replay buffer: {json.dumps(stats, indent=2)}')

## Phase 3: Train JEPA-Lite World Model

Train the transition predictor to predict latent next state
and auxiliary heads for validity, score gain, compute cost, repair need.

In [ ]:
from torch.utils.data import DataLoader
from v4_1_reasoning_system.training.train_world_model import (
    WorldModelTrainer,
    WorldModelDataset,
    build_world_model_dataset,
)

# Build dataset from replay buffer
wm_records = build_world_model_dataset(
    controller.replay_buffer,
    latent_dim=LATENT_DIM,
    action_dim=ACTION_DIM,
)
print(f'World model dataset: {len(wm_records)} records')

if len(wm_records) < 10:
    print('WARNING: Too few records. Collect more rollouts or use data augmentation.')

In [ ]:
# Create data loaders
if len(wm_records) >= 10:
    from torch.utils.data import random_split
    
    dataset = WorldModelDataset(wm_records)
    val_size = max(1, int(len(dataset) * 0.2))
    train_size = len(dataset) - val_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
    
    print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# Train world model
if len(wm_records) >= 10:
    wm_trainer = WorldModelTrainer(
        predictor=controller.transition_predictor,
        aux_heads=controller.aux_heads,
        lr=3e-4,
        device=device,
    )
    
    NUM_EPOCHS = 100
    for epoch in range(NUM_EPOCHS):
        train_m = wm_trainer.train_epoch(train_loader)
        if (epoch + 1) % 20 == 0:
            val_m = wm_trainer.evaluate(val_loader)
            print(
                f'Epoch {epoch+1:3d}/{NUM_EPOCHS}: '
                f'loss={train_m["loss"]:.4f} latent={train_m["latent_loss"]:.4f} '
                f'aux={train_m["aux_loss"]:.4f} | '
                f'val_latent={val_m["val_latent_loss"]:.4f} '
                f'val_validity_acc={val_m["val_validity_acc"]:.3f}'
            )
    
    print('World model training complete.')
else:
    print('Skipping world model training (insufficient data).')

## Phase 4: Train Router EBM

Train the Energy-Based Model router using ranking loss on
good vs bad reasoning actions from the collected trajectories.

In [ ]:
from v4_1_reasoning_system.training.train_router import train_router_from_buffer

router_result = train_router_from_buffer(
    router=controller.ebm_router,
    replay_buffer=controller.replay_buffer,
    num_epochs=100,
    batch_size=16,
    lr=1e-4,
    margin=1.0,
    device=device,
    verbose=True,
)

print(f'\nRouter training result: {router_result["status"]}')
if router_result['status'] == 'trained':
    print(f'Best val accuracy: {router_result["best_val_accuracy"]:.3f}')
    print(f'Training pairs: {router_result["num_pairs"]}')

## Phase 5: Evaluate with Learned Router

Switch to the trained EBM router and re-run the benchmark.

In [ ]:
# Switch to learned router
controller.use_learned_router = True

# Generate fresh test problems
test_problems = RolloutCollector.generate_benchmark_suite(
    n_planning=10,
    n_scheduling=10,
    n_optimization=10,
    n_coding=0,
)

# Evaluate
test_results = collector.collect(test_problems, verbose=True)

test_successes = sum(1 for r in test_results if r['valid'])
print(f'\nLearned router: {test_successes}/{len(test_results)} valid')
print(f'Avg score: {sum(r["score"] for r in test_results) / len(test_results):.3f}')

In [ ]:
# Compare rule-based vs learned router
rule_scores = [r['score'] for r in results]
learned_scores = [r['score'] for r in test_results]

print(f'Rule-based  — avg score: {sum(rule_scores)/len(rule_scores):.3f}, '
      f'success: {sum(1 for r in results if r["valid"])}/{len(results)}')
print(f'Learned EBM — avg score: {sum(learned_scores)/len(learned_scores):.3f}, '
      f'success: {test_successes}/{len(test_results)}')

## Save Final Checkpoint

In [ ]:
CHECKPOINT_DIR = '/content/drive/MyDrive/agi/checkpoints/v4_1_trained'
controller.save_checkpoint(CHECKPOINT_DIR)
print(f'Full checkpoint saved to {CHECKPOINT_DIR}')

# List saved files
for p in sorted(Path(CHECKPOINT_DIR).rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f'  {p.relative_to(CHECKPOINT_DIR)}: {size_mb:.2f} MB')

## Load Checkpoint & Resume

To resume training or run inference from a saved checkpoint:

In [ ]:
# Create a fresh controller and load weights
controller2 = ReasoningController(
    use_learned_router=True,
    max_iterations=5,
    device=device,
    latent_dim=LATENT_DIM,
    action_dim=ACTION_DIM,
)
controller2.load_checkpoint(CHECKPOINT_DIR)
print('Checkpoint loaded successfully')
print(f'Replay buffer: {len(controller2.replay_buffer)} trajectories')